In [117]:
import seaborn as sns
import pandas as pd
import numpy as np
import optuna
import matplotlib.pyplot as plt
from sklearn import datasets

In [118]:
df = pd.read_csv('used_cars.csv')


In [119]:
df ['price'] = df['price'].str.replace('$', '')

In [120]:
df ['price'] = df['price'].str.replace(',', '')

In [121]:
df['price'] = df['price'].astype(float)

In [122]:
df ['milage'] = df['milage'].str.replace('mi.', '')

In [123]:
df = df.drop('clean_title', axis=1)

In [124]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4009 entries, 0 to 4008
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   brand         4009 non-null   object 
 1   model         4009 non-null   object 
 2   model_year    4009 non-null   int64  
 3   milage        4009 non-null   object 
 4   fuel_type     3839 non-null   object 
 5   engine        4009 non-null   object 
 6   transmission  4009 non-null   object 
 7   ext_col       4009 non-null   object 
 8   int_col       4009 non-null   object 
 9   accident      3896 non-null   object 
 10  price         4009 non-null   float64
dtypes: float64(1), int64(1), object(9)
memory usage: 344.7+ KB


In [125]:
df.isna().sum()


brand             0
model             0
model_year        0
milage            0
fuel_type       170
engine            0
transmission      0
ext_col           0
int_col           0
accident        113
price             0
dtype: int64

In [126]:
df['accident'].fillna('None Reported', inplace=True)

In [127]:
df = df.dropna(subset=['fuel_type'])

In [128]:
df ['fuel_type'] = df['fuel_type'].str.replace('–', '')

In [129]:
df ['engine'] = df['engine'].str.replace('–', '')

In [130]:
df.dropna(inplace=True)


In [131]:
df.head()

,brand,model,model_year,milage,fuel_type,engine,transmission,ext_col,int_col,accident,price
0,Ford,Utility Police Interceptor Base,2013,"51,000",E85 Flex Fuel,300.0HP 3.7L V6 Cylinder Engine Flex Fuel Capa...,6-Speed A/T,Black,Black,At least 1 accident or damage reported,10300.0
1,Hyundai,Palisade SEL,2021,"34,742",Gasoline,3.8L V6 24V GDI DOHC,8-Speed Automatic,Moonlight Cloud,Gray,At least 1 accident or damage reported,38005.0
2,Lexus,RX 350 RX 350,2022,"22,372",Gasoline,3.5 Liter DOHC,Automatic,Blue,Black,None reported,54598.0
3,INFINITI,Q50 Hybrid Sport,2015,"88,900",Hybrid,354.0HP 3.5L V6 Cylinder Engine Gas/Electric H...,7-Speed A/T,Black,Black,None reported,15500.0
4,Audi,Q3 45 S line Premium Plus,2021,"9,835",Gasoline,2.0L I4 16V GDI DOHC Turbo,8-Speed Automatic,Glacier White Metallic,Black,None reported,34999.0


In [132]:
# Normalizamos la columna engine a string en minúsculas
engine_str = df['engine'].astype(str).str.lower()
liters = (
    engine_str
    .str.extract(r'(\d+[.,]?\d*)\s*l')[0]
    .str.replace(',', '.', regex=False)
    .astype(float)
)

liters = np.where(liters > 10, liters / 10, liters)
df['engine_liters'] = pd.Series(liters).fillna(-1)
df['cylinders'] = (
    engine_str
    .str.extract(r'(\d+)\s*cylinder|v(\d+)')
    .bfill(axis=1)
    .iloc[:, 0]
)
df['cylinders'] = pd.to_numeric(
    df['cylinders'],
    errors='coerce'
).fillna(-1)
df['turbo'] = engine_str.str.contains('turbo', na=False)
df['horsepower'] = (
    engine_str
    .str.extract(r'(\d+[.,]?\d*)\s*hp')[0]
    .str.replace(',', '.', regex=False)
    .astype(float)
    .fillna(-1)
    .astype(int)
)
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3839 entries, 0 to 4008
Data columns (total 15 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   brand          3839 non-null   object 
 1   model          3839 non-null   object 
 2   model_year     3839 non-null   int64  
 3   milage         3839 non-null   object 
 4   fuel_type      3839 non-null   object 
 5   engine         3839 non-null   object 
 6   transmission   3839 non-null   object 
 7   ext_col        3839 non-null   object 
 8   int_col        3839 non-null   object 
 9   accident       3839 non-null   object 
 10  price          3839 non-null   float64
 11  engine_liters  3676 non-null   float64
 12  cylinders      3839 non-null   float64
 13  turbo          3839 non-null   bool   
 14  horsepower     3839 non-null   int64  
dtypes: bool(1), float64(3), int64(2), object(9)
memory usage: 453.6+ KB


In [133]:
df = df.dropna()
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3676 entries, 0 to 3838
Data columns (total 15 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   brand          3676 non-null   object 
 1   model          3676 non-null   object 
 2   model_year     3676 non-null   int64  
 3   milage         3676 non-null   object 
 4   fuel_type      3676 non-null   object 
 5   engine         3676 non-null   object 
 6   transmission   3676 non-null   object 
 7   ext_col        3676 non-null   object 
 8   int_col        3676 non-null   object 
 9   accident       3676 non-null   object 
 10  price          3676 non-null   float64
 11  engine_liters  3676 non-null   float64
 12  cylinders      3676 non-null   float64
 13  turbo          3676 non-null   bool   
 14  horsepower     3676 non-null   int64  
dtypes: bool(1), float64(3), int64(2), object(9)
memory usage: 434.4+ KB


In [134]:
df ['accident'] = df['accident'].str.replace('None Reported', 'None reported')

In [135]:
df.head()

,brand,model,model_year,milage,fuel_type,engine,transmission,ext_col,int_col,accident,price,engine_liters,cylinders,turbo,horsepower
0,Ford,Utility Police Interceptor Base,2013,"51,000",E85 Flex Fuel,300.0HP 3.7L V6 Cylinder Engine Flex Fuel Capa...,6-Speed A/T,Black,Black,At least 1 accident or damage reported,10300.0,3.7,6.0,False,300
1,Hyundai,Palisade SEL,2021,"34,742",Gasoline,3.8L V6 24V GDI DOHC,8-Speed Automatic,Moonlight Cloud,Gray,At least 1 accident or damage reported,38005.0,3.8,6.0,False,-1
2,Lexus,RX 350 RX 350,2022,"22,372",Gasoline,3.5 Liter DOHC,Automatic,Blue,Black,None reported,54598.0,3.5,-1.0,False,-1
3,INFINITI,Q50 Hybrid Sport,2015,"88,900",Hybrid,354.0HP 3.5L V6 Cylinder Engine Gas/Electric H...,7-Speed A/T,Black,Black,None reported,15500.0,3.5,6.0,False,354
4,Audi,Q3 45 S line Premium Plus,2021,"9,835",Gasoline,2.0L I4 16V GDI DOHC Turbo,8-Speed Automatic,Glacier White Metallic,Black,None reported,34999.0,2.0,-1.0,True,-1


In [136]:
cols_to_string = ['brand','model','fuel_type','engine','transmission','ext_col','int_col','accident']
df[cols_to_string] = df[cols_to_string].astype('string')
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3676 entries, 0 to 3838
Data columns (total 15 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   brand          3676 non-null   string 
 1   model          3676 non-null   string 
 2   model_year     3676 non-null   int64  
 3   milage         3676 non-null   object 
 4   fuel_type      3676 non-null   string 
 5   engine         3676 non-null   string 
 6   transmission   3676 non-null   string 
 7   ext_col        3676 non-null   string 
 8   int_col        3676 non-null   string 
 9   accident       3676 non-null   string 
 10  price          3676 non-null   float64
 11  engine_liters  3676 non-null   float64
 12  cylinders      3676 non-null   float64
 13  turbo          3676 non-null   bool   
 14  horsepower     3676 non-null   int64  
dtypes: bool(1), float64(3), int64(2), object(1), string(8)
memory usage: 434.4+ KB


In [137]:
df['milage'] = df['milage'].str.replace(',','').astype('float64')
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3676 entries, 0 to 3838
Data columns (total 15 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   brand          3676 non-null   string 
 1   model          3676 non-null   string 
 2   model_year     3676 non-null   int64  
 3   milage         3676 non-null   float64
 4   fuel_type      3676 non-null   string 
 5   engine         3676 non-null   string 
 6   transmission   3676 non-null   string 
 7   ext_col        3676 non-null   string 
 8   int_col        3676 non-null   string 
 9   accident       3676 non-null   string 
 10  price          3676 non-null   float64
 11  engine_liters  3676 non-null   float64
 12  cylinders      3676 non-null   float64
 13  turbo          3676 non-null   bool   
 14  horsepower     3676 non-null   int64  
dtypes: bool(1), float64(4), int64(2), string(8)
memory usage: 434.4 KB


In [138]:
missing_values = df.isnull().sum().sort_values(ascending=False)
print(missing_values)

brand            0
model            0
model_year       0
milage           0
fuel_type        0
engine           0
transmission     0
ext_col          0
int_col          0
accident         0
price            0
engine_liters    0
cylinders        0
turbo            0
horsepower       0
dtype: int64


In [139]:
df.to_csv("data/clean_data.csv", index=False)